# Week 5 · Day 4 — Daily Challenge
## Strategic Analysis of Superstore Performance

**Business objective:** identify revenue drivers, geographic concentration, profitable products, and potential discount-related losses using the US Superstore dataset.

**Source:** [Kaggle — Superstore Dataset Final](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final). Download and unzip the dataset; put `Sample - Superstore.csv` (or another CSV from that download) in the same folder as this notebook. This notebook does not require Kaggle API credentials.

**Important:** These are observational sales records. Correlations between discounts and profits do not establish that discounts *caused* changes in profit.

### 1. Imports and data preparation

In [ ]:
import warnings
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)


In [ ]:
# Place the Kaggle CSV beside this notebook, or update DATA_PATH explicitly.
DATA_PATH = None  # e.g. Path('Sample - Superstore.csv')
if DATA_PATH is None:
    candidates = sorted(Path('.').glob('*.csv'))
    if not candidates:
        raise FileNotFoundError('Download the Kaggle Superstore CSV and place it beside the notebook.')
    DATA_PATH = candidates[0]
print('Loading:', DATA_PATH)
try:
    raw = pd.read_csv(DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    raw = pd.read_csv(DATA_PATH, encoding='latin-1')
print('Original shape:', raw.shape)
print('Columns:', raw.columns.tolist())
display(raw.head())
raw.info()
display(raw.describe(include='all').T)
display(raw.isna().sum().rename('Missing values').to_frame())
print('Exact duplicate rows:', raw.duplicated().sum())

**Cleaning rationale:** Remove exact duplicate records to avoid double-counting; retain missing postal codes as missing because substituting `0` would create a fictitious ZIP code. Exclude rows missing essential sales/profit/date/category/state/product fields rather than inventing business transactions. Coerce invalid numeric values to missing, and parse dates explicitly. Sales of zero are retained but row-level profit margins are undefined in that case. Date parsing is month-first to match the US source.

In [ ]:
df = raw.drop_duplicates().copy()
required = ['Order Date', 'Ship Date', 'Sales', 'Profit', 'Discount', 'Category', 'State', 'Product Name']
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise KeyError(f'Required columns missing: {missing_cols}')
for c in ['Order Date', 'Ship Date']:
    df[c] = pd.to_datetime(df[c], errors='coerce', dayfirst=False)
for c in ['Sales', 'Profit', 'Discount']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
if 'Postal Code' in df.columns:
    df['Postal Code'] = df['Postal Code'].astype('string').str.replace(r'\.0$', '', regex=True)
before = len(df)
df = df.dropna(subset=['Order Date', 'Sales', 'Profit', 'Discount', 'Category', 'State', 'Product Name']).copy()
print('Rows dropped for missing essential fields:', before - len(df))
if df.empty:
    raise ValueError('No usable records remain after cleaning.')
df['Profit Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'] * 100, np.nan)
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')
print('Cleaned shape:', df.shape)
display(df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head())
display(df.isna().sum().rename('Remaining missing values').to_frame())

### 2. Key performance indicators and trends

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_margin = 100 * total_profit / total_sales if total_sales else np.nan
print(f'Total sales: ${total_sales:,.2f}')
print(f'Total profit: ${total_profit:,.2f}')
print(f'Weighted overall profit margin: {overall_margin:.2f}%')
print(f'Unique orders: {df["Order ID"].nunique():,}' if 'Order ID' in df else f'Line items: {len(df):,}')
monthly_total = df.groupby('Order Month-Year')['Sales'].sum().sort_index()
monthly_sales = df.groupby(['Order Month-Year', 'Category'], observed=True)['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_total.index.to_timestamp(), monthly_total.values, marker='o', markersize=3)
ax.set(title='Monthly sales — all categories', xlabel='Order month', ylabel='Sales ($)')
fig.autofmt_xdate(); plt.tight_layout(); plt.show()


**Interactive category selector:** Choose *All* or an individual product category. Run this notebook in Jupyter with `ipywidgets` enabled to use the control.

In [ ]:
def plot_monthly_sales(category='All'):
    if category == 'All':
        series = monthly_total
        dates, values = series.index.to_timestamp(), series.values
    else:
        sub = monthly_sales[monthly_sales['Category'] == category]
        dates, values = sub['Date'], sub['Sales']
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(dates, values, marker='o', markersize=3)
    ax.set(title=f'Monthly sales — {category}', xlabel='Order month', ylabel='Sales ($)')
    fig.autofmt_xdate(); plt.tight_layout(); plt.show()

category_dropdown = widgets.Dropdown(options=['All'] + sorted(df['Category'].dropna().unique().tolist()), value='All', description='Category:')
widgets.interact(plot_monthly_sales, category=category_dropdown);


### 3. Geographic sales analysis

In [ ]:
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
print('States represented:', len(state_sales))
display(state_sales.head(10).rename('Sales ($)').to_frame())

def plot_top_states(top_n=10):
    top = state_sales.head(top_n).sort_values()
    fig, ax = plt.subplots(figsize=(12, max(5, 0.42 * len(top))))
    bars = ax.barh(top.index, top.values)
    ax.bar_label(bars, labels=[f'${v:,.0f}' for v in top.values], padding=3, fontsize=8)
    ax.set_xlim(0, max(top.values) * 1.20)
    ax.set(title=f'Top {top_n} states by sales', xlabel='Sales ($)', ylabel='State')
    plt.tight_layout(); plt.show()
    print(f'Top {top_n} states: ${top.sum():,.0f}, {top.sum()/total_sales:.1%} of sales')

widgets.interact(plot_top_states, top_n=widgets.IntSlider(value=min(10, len(state_sales)), min=1, max=len(state_sales), step=1, description='Top N:'));


### 4. Top ten most profitable products

In [ ]:
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False)
top_products = product_profit.head(10).sort_values()
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=top_products.values, y=top_products.index, color='teal', ax=ax)
for bar, value in zip(ax.patches, top_products.values):
    ax.annotate(f'${value:,.0f}', (bar.get_width(), bar.get_y() + bar.get_height()/2), xytext=(5,0), textcoords='offset points', va='center', fontsize=9)
ax.set_xlim(0, max(top_products.values) * 1.25)
ax.set(title='Executive view: 10 products generating the most profit', xlabel='Total profit ($)', ylabel='Product')
plt.tight_layout(); plt.show()
print(f'Top product: {product_profit.index[0]} — ${product_profit.iloc[0]:,.2f}')
print(f'Top 10 combined profit: ${product_profit.head(10).sum():,.2f}')


### 5. Discount diagnostics and profitability

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=.45, s=22, ax=ax)
sns.regplot(data=df, x='Discount', y='Profit', scatter=False, color='black', line_kws={'linestyle':'--'}, ax=ax)
ax.axhline(0, color='gray', linewidth=1)
ax.set(title='Discount versus transaction-line profit by category', xlabel='Discount rate', ylabel='Profit ($)')
plt.tight_layout(); plt.show()

high = df[df['Discount'] > .20]
low = df[df['Discount'] <= .20]
print('High-discount (>20%) rows:', len(high))
print('High-discount mean profit:', f'${high["Profit"].mean():,.2f}' if len(high) else 'Not available')
print('High-discount loss rate:', f'{(high["Profit"] < 0).mean():.1%}' if len(high) else 'Not available')
print('Lower-discount loss rate:', f'{(low["Profit"] < 0).mean():.1%}' if len(low) else 'Not available')
discount_by_category = df.assign(High_Discount=df['Discount'] > .20).groupby(['Category','High_Discount']).agg(rows=('Profit','size'), avg_profit=('Profit','mean'), loss_rate=('Profit', lambda x: (x < 0).mean())).reset_index()
display(discount_by_category)


**Interpretation caution:** A 20% threshold is a useful *diagnostic split*, not a proven optimal discount cap. Product mix, transaction size, and region can confound the observed relationship. Test policy changes with a controlled pilot before imposing a universal rule.

### 6. Matplotlib versus Seaborn — methodology

| Criterion | Matplotlib | Seaborn |
|---|---|---|
| Strength | Fine-grained chart and annotation control | Concise statistical and category-aware plots |
| Interactivity | Integrates with `ipywidgets` | Also integrates with `ipywidgets` via Matplotlib |
| Best use here | Monthly trends, state rankings, dashboard layout | Product ranking, scatter/regression diagnostics |
| Trade-off | More manual styling | Some custom details require Matplotlib calls |

**Recommendation:** Use Matplotlib for customized executive dashboard layouts and widget callbacks; use Seaborn for rapid statistical exploration and polished category comparisons. Neither library is universally faster; rendering performance depends on data size, plot type, and environment.

In [ ]:
yearly = df.groupby('Order Year', as_index=False)['Sales'].sum()
t0 = time.perf_counter()
fig, ax = plt.subplots(); ax.plot(yearly['Order Year'], yearly['Sales']); plt.close(fig)
matplotlib_time = time.perf_counter() - t0
t0 = time.perf_counter()
fig, ax = plt.subplots(); sns.lineplot(data=yearly, x='Order Year', y='Sales', ax=ax); plt.close(fig)
seaborn_time = time.perf_counter() - t0
print(f'Matplotlib render setup: {matplotlib_time:.4f}s')
print(f'Seaborn render setup: {seaborn_time:.4f}s')
print('Single-run timings are illustrative, not a reliable benchmark.')

### 7. Optional: four-panel executive dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes[0,0].plot(monthly_total.index.to_timestamp(), monthly_total.values)
axes[0,0].set_title('Monthly sales')
axes[0,1].bar(df.groupby('Category')['Sales'].sum().index, df.groupby('Category')['Sales'].sum().values)
axes[0,1].set_title('Sales by category')
top10 = state_sales.head(10).sort_values()
axes[1,0].barh(top10.index, top10.values)
axes[1,0].set_title('Top 10 states by sales')
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', s=10, alpha=.35, ax=axes[1,1])
axes[1,1].axhline(0, color='gray', linewidth=1)
axes[1,1].set_title('Discount versus profit')
fig.suptitle('Superstore executive dashboard', fontsize=16)
fig.tight_layout(); plt.show()

### 8. Optional: annotate high- and low-profit outliers

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category', alpha=.4, s=20, ax=ax)
outliers = pd.concat([df.nlargest(3, 'Profit'), df.nsmallest(3, 'Profit')]).drop_duplicates()
for _, row in outliers.iterrows():
    ax.annotate(f'${row["Profit"]:,.0f}', (row['Discount'], row['Profit']), xytext=(6,6), textcoords='offset points', fontsize=8)
ax.set(title='Discount versus profit — annotated extremes', xlabel='Discount', ylabel='Profit ($)')
plt.tight_layout(); plt.show()

### 9. Optional: Plotly interactive scatter (if installed)

In [ ]:
try:
    import plotly.express as px
    fig = px.scatter(df, x='Discount', y='Profit', color='Category', hover_data=['Product Name', 'Sales'], title='Interactive discount versus profit')
    fig.show()
    print('Plotly offers hover, zoom, and pan; Matplotlib + ipywidgets offers custom notebook controls.')
except ImportError:
    print('Optional Plotly chart skipped. Install with: py -m pip install plotly')

### 10. Executive summary and strategic recommendations

In [ ]:
top_state = state_sales.index[0]
top_category = df.groupby('Category')['Sales'].sum().idxmax()
top5_share = state_sales.head(5).sum() / total_sales
high_loss = (high['Profit'] < 0).mean() if len(high) else np.nan
low_loss = (low['Profit'] < 0).mean() if len(low) else np.nan

def fmt_rate(x):
    return 'N/A' if pd.isna(x) else f'{x:.1%}'

summary = f"""## Executive summary

1. **Business performance:** ${total_sales:,.0f} in recorded sales, ${total_profit:,.0f} profit, and an overall weighted profit margin of {overall_margin:.1f}%.
2. **Geography:** {top_state} has the highest recorded sales (${state_sales.iloc[0]:,.0f}); the top five states contribute {top5_share:.1%} of sales. Investigate concentration and expansion opportunities.
3. **Products:** {top_category} is the largest category by sales; the most profitable named product is **{product_profit.index[0]}** (${product_profit.iloc[0]:,.0f} total profit). Prioritize margin-aware product planning.
4. **Discounts:** The observed loss rate is {fmt_rate(high_loss)} for discounts above 20%, versus {fmt_rate(low_loss)} at or below 20%. Review high-discount transactions by category before changing pricing policy.
5. **Next steps:** Monitor monthly trends, assess regional profitability (not just revenue), and test discount-policy changes with controlled experiments.

**Limitations:** This historical dataset does not by itself establish causality, future forecasts, or an optimal discount threshold. Record-level margin averages are not equivalent to the weighted business profit margin.
"""
display(Markdown(summary))


### 11. Save reproducible analysis outputs

In [ ]:
out = Path('results')
out.mkdir(exist_ok=True)
monthly_total.rename('Sales').to_csv(out / 'monthly_sales.csv')
state_sales.rename('Sales').to_csv(out / 'state_sales.csv')
product_profit.head(10).rename('Profit').to_csv(out / 'top_10_products.csv')
discount_by_category.to_csv(out / 'discount_by_category.csv', index=False)
(out / 'executive_summary.md').write_text(summary, encoding='utf-8')
print('Saved supporting tables and executive summary to:', out.resolve())